In [0]:
%run "../../utils/00_config"

In [0]:
%run "../../utils/02_data_quality"

In [0]:
from pyspark.sql import functions as F

stg_orders = spark.table(
    get_table("staging", "stg_wfm_orders")
)

In [0]:
source_count = stg_orders.count()

print(f"Source table: {get_table('staging', 'stg_wfm_orders')}")
print(f"Total records: {source_count}")

check_unique_keys(
    stg_orders,
    ["execution_id", "order_id"]
)

print("Source validation passed.")

In [0]:
# Build fact_order_observation

fact_order_observation = stg_orders.select(
    "execution_id",
    "observed_at",
    "product_slug",
    "item_slug",
    "order_id",
    "item_id",
    "order_type",
    "platinum",
    "quantity",
    "per_trade",
    F.col("is_visible").alias("visible"),
    "order_created_at",
    "order_updated_at",
    "player_id",
    "player_name",
    "player_slug",
    "player_reputation",
    "platform",
    "crossplay",
    "locale",
    "player_status",
    "player_activity_type",
    "player_activity_details",
    "player_last_seen"
)

In [0]:
# Data quality checks
check_required_fields(
    fact_order_observation,
    [
        "execution_id",
        "observed_at",
        "product_slug",
        "item_slug",
        "order_id",
        "item_id",
        "order_type",
        "player_id"
    ]
)

check_unique_keys(
    fact_order_observation,
    ["execution_id", "order_id"]
)

print(f"Fact rows: {fact_order_observation.count()}")
print("Fact table validations passed.")

# Referential integrity checks

dim_item = spark.table(
    get_table("intermediate", "dim_item")
)

dim_player = spark.table(
    get_table("intermediate", "dim_player")
)

missing_items = (
    fact_order_observation
    .select("item_id")
    .distinct()
    .join(
        dim_item.select("item_id").distinct(),
        on="item_id",
        how="left_anti"
    )
)

missing_players = (
    fact_order_observation
    .select("player_id")
    .distinct()
    .join(
        dim_player.select("player_id").distinct(),
        on="player_id",
        how="left_anti"
    )
)

assert missing_items.isEmpty(), "Fact contains unknown item IDs."
assert missing_players.isEmpty(), "Fact contains unknown player IDs."

print("Referential integrity checks passed.")

In [0]:
# Materialize fact_order_observation

from delta.tables import DeltaTable

target_table = get_table(
    "intermediate",
    "fact_order_observation"
)

merge_condition = """
    target.execution_id = source.execution_id
    AND target.order_id = source.order_id
"""

if spark.catalog.tableExists(target_table):
    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(
            fact_order_observation.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        fact_order_observation.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

print(f"Table materialized: {target_table}")